# 🧠 Project: Impact of Short-Term Rentals on the Housing Market
This notebook is part of Phase 1 of the project for the course Information Integration and Analytic Processing.

📌 Objectives:
- Integrate data from multiple sources.
- Analyze the relationship between short-term rental density, population, and rental prices.
- Implement data cleaning, schema integration, and identity resolution techniques.

---

## 1.📦 Data Acquisition

### 1.1 Dataset Overview

In this section, we import the different datasets required to analyze the impact of short-term rentals on urban areas in Portugal. Each dataset comes from a distinct source and provides a specific perspective on urban characteristics, such as population density, rental supply, housing prices, and administrative divisions.

The datasets imported are as follows:

- **`portugal_listings.csv`**: Contains information about properties available for purchase across the country, including fields like `Price`, `District`, `City`, and `Town`.  
  **Source**: https://www.kaggle.com/datasets/luvathoms/portugal-real-estate-2024

- **`rendasm2.json`**: Provides average rental prices per square meter for different regions in **2023**.  
  **Source**: https://www.ine.pt/bddXplorer/htdocs/minfo.jsp?var_cd=0012600&lingua=PT

- **`densidadePopulacional.json`**: Indicates population density (inhabitants/km²) by municipality for **2023**.  
  **Source**: https://www.ine.pt/bddXplorer/htdocs/minfo.jsp?var_cd=0008337&lingua=PT

- **`densidadealojamentosm2.json`**: Presents the density of short-term rental units (units/km²) at the **parish level**, for **2021**.  
  **Source**: https://www.ine.pt/bddXplorer/htdocs/minfo.jsp?var_cd=0012514&lingua=PT

- **`Areas_Freg_Conc_Dist_Pais_CAOP2019.xls`**: The CAOP (Carta Administrativa Oficial de Portugal) file defines the country’s administrative structure, including districts, municipalities, and parishes. It is used as a reference to unify geographic codes (`DICO`, `geocod`, etc.).  
  **Source**: https://www.dgterritorio.gov.pt/cartografia/cartografia-tematica/caop

> ⚠️ Each file was loaded based on its internal structure (`.csv`, `.json`, `.xls`) and only the relevant content was extracted for further cleaning and integration.

### 1.2 Code

In [ ]:
import pandas as pd

# Load dataset with property listings available for sale
property_listings = pd.read_csv(r'dados\portugal_listings.csv')

# Load dataset with average rental prices per square meter for 2023
rental_prices_raw = pd.read_json(r'dados\rendasm2.json')
rental_prices = pd.DataFrame(rental_prices_raw['Dados'][0]['2023'])

# Load dataset with population density (inhabitants/km²) by municipality for 2023
population_density_raw = pd.read_json(r'dados\densidadePopulacional.json')
population_density = pd.DataFrame(population_density_raw['Dados'][0]['2023'])

# Load dataset with short-term rental density (units/km²) by parish for 2021
short_term_rental_density_raw = pd.read_json(r'dados\densidadealojamentosm2.json')
short_term_rental_density = pd.DataFrame(short_term_rental_density_raw['Dados'][0]['2021'])

# Load reference table with official administrative boundaries (CAOP)
municipalities = pd.read_excel(r'dados\Areas_Freg_Conc_Dist_Pais_CAOP2019.xls',
                               sheet_name='Areas_Concelhos_CAOP2019')


All datasets are now successfully loaded into memory and stored in well-labeled variables.  
In the next section, we will perform data profiling and initial integration in order to understand their structure and prepare for cleaning and transformation.

---

## 2. 🔍 Data Profiling

### 2.1 Overview

After importing the data, an exploratory analysis (*data profiling*) was conducted to understand the structure and quality of each dataset. This included:

- Data type identification  
- Counting and percentage of missing values  
- Number of unique values per column (*cardinality*)

A custom function was used to display these metrics for each dataset.

In [ ]:
def data_profiling(df: pd.DataFrame):
    profiling = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.values,
        "Non-Null Count": df.notnull().sum().values,
        "Null Count": df.isnull().sum().values,
        "% Nulls": (df.isnull().mean() * 100).round(2).values,
        "Unique Values": df.nunique().values
    })
    display(profiling)


### 2.2 Profiling by Dataset

#### 2.2.1 Administrative Boundaries (`municipalities`)

- **Granularity**: Municipality-level  
- **Relevant columns**:
  - `DICO`: unique identifier → used as `MunicipalityCode`
  - `CONCELHO_DSG`: municipality name
  - `NUTSIII_DSG`: NUTS III region

- Ignored columns:
  - Other NUTS codes, area, perimeter, altitude


In [ ]:
data_profiling(municipalities)

,Coluna,Tipo de Dado,Valores Não Nulos,Valores Nulos,% Nulos,Valores Únicos
0,DICO,int64,308,0,0.0,308
1,NUTSI_DSG,object,308,0,0.0,3
2,NUTSI_COD,int64,308,0,0.0,3
3,NUTSII_DSG,object,308,0,0.0,7
4,NUTSII_COD,int64,308,0,0.0,7
5,NUTSIII_DSG,object,308,0,0.0,25
6,NUTSIII_COD,object,308,0,0.0,25
7,DISTRITO_ILHA_DSG,object,308,0,0.0,29
8,CONCELHO_DSG,object,308,0,0.0,307
9,AREA_2019_ha,float64,308,0,0.0,308


#### 2.2.2 Short-Term Rental Density (`short_term_rental_density`)

- **Granularity**: Parish-level (freguesia)  
- **Relevant columns**:
  - `geocod`: used to derive `MunicipalityCode` (first 4 digits)
  - `geodsg`: parish name → used to infer `Municipality`
  - `valor`: indicator value → converted to float

- Ignored columns:
  - `ind_string`


In [ ]:
data_profiling(short_term_rental_density)

,Coluna,Tipo de Dado,Valores Não Nulos,Valores Nulos,% Nulos,Valores Únicos
0,geocod,object,3439,0,0.0,3439
1,geodsg,object,3439,0,0.0,3075
2,ind_string,object,3439,0,0.0,1656
3,valor,object,3439,0,0.0,1656


#### 2.2.3 Population Density (`population_density`)

- **Granularity**: Municipality-level  
- **Relevant columns**:
  - `geocod`: used to derive `MunicipalityCode` (last 4 digits)
  - `geodsg`: municipality name
  - `valor`: population density → converted to float

- Ignored columns:
  - `ind_string`


In [ ]:
data_profiling(population_density)

,Coluna,Tipo de Dado,Valores Não Nulos,Valores Nulos,% Nulos,Valores Únicos
0,geocod,object,347,0,0.0,347
1,geodsg,object,347,0,0.0,340
2,ind_string,object,347,0,0.0,316
3,valor,object,347,0,0.0,316


#### 2.2.4 Property Listings (`property_listings`)

- **Granularity**: Municipality-level (inferred from city names)  
- **Relevant columns**:
  - `Price`: property sale price
  - `District`: administrative district
  - `City`: municipality inferred from city name

- Ignored columns:
  - `Town`
  - All columns starting with `Unnamed:` (present due to CSV formatting)

In [ ]:
data_profiling(property_listings)

,Coluna,Tipo de Dado,Valores Não Nulos,Valores Nulos,% Nulos,Valores Únicos
0,Price,float64,135236,300,0.22,4754
1,District,object,135536,0,0.00,27
2,City,object,135536,0,0.00,275
3,Town,object,135534,2,0.00,2263
4,Unnamed: 4,float64,0,135536,100.00,0
5,Unnamed: 5,float64,0,135536,100.00,0
6,Unnamed: 6,float64,0,135536,100.00,0
7,Unnamed: 7,float64,0,135536,100.00,0
8,Unnamed: 8,float64,0,135536,100.00,0
9,Unnamed: 9,float64,0,135536,100.00,0


#### 2.2.5 Rental Prices (`rental_prices`)

- **Granularity**: Parish-level (freguesia)  
- **Relevant columns**:
  - `geocod`: used to derive `MunicipalityCode` (last 4 digits)
  - `geodsg`: name used as `Municipality`
  - `valor`: average rent → converted to float

- Ignored columns:
  - `ind_string`, `sinal_conv`, `sinal_conv_desc`


In [ ]:
data_profiling(rental_prices)

,Coluna,Tipo de Dado,Valores Não Nulos,Valores Nulos,% Nulos,Valores Únicos
0,geocod,object,638,0,0.00,638
1,geodsg,object,638,0,0.00,616
2,ind_string,object,638,0,0.00,343
3,valor,object,431,207,32.45,341
4,sinal_conv,object,207,431,67.55,2
5,sinal_conv_desc,object,207,431,67.55,2


### 2.3 Key Identifiers Summary

| Dataset                    | Primary Key                             | Other Relevant Columns                      |
|----------------------------|------------------------------------------|---------------------------------------------|
| `municipalities`           | `DICO` → `MunicipalityCode`             | `CONCELHO_DSG` → `Municipality`, `NUTSIII_DSG` → `Region` |
| `population_density`       | `geocod[-4:]` → `MunicipalityCode`      | `geodsg` → `Municipality`                   |
| `short_term_rental_density`| `geocod[:4]` → `MunicipalityCode`       | `geodsg` → `Municipality`                   |
| `rental_prices`            | `geocod[-4:]` → `MunicipalityCode`      | `geodsg` → `Municipality`                   |
| `property_listings`        | `City`                                   | `District`, `Price`                         |

> **Note:** The `geocod` field encodes geographic hierarchy:
> - In **parish-level** datasets (like `short_term_rental_density`), the **first 4 digits** of `geocod` identify the municipality.
> - In **municipality-level** datasets (like `population_density`), the **last 4 digits** of `geocod` represent the municipality code directly.
>
> **Example:**
> - `geocod = "150819"` in a parish-level dataset → `geocod[:4] = "1508"` (MunicipalityCode)
> - `geocod = "171508"` in a municipality-level dataset → `geocod[-4:] = "1508"` (MunicipalityCode)

This adjustment ensures all datasets can be joined consistently on the same spatial level.


### 2.4 Cleaning and Normalization Plan

Based on the data profiling results, the following cleaning and normalization actions will be applied in the next phase:

- Convert all `valor` columns to float  
- Remove rows with null `Price` in `property_listings`  
- Fill missing `valor` values in `rental_prices` using municipality-level averages  
- Normalize all location-related text fields:
  - Remove accents
  - Convert to lowercase
  - Trim whitespace
  - Create new fields: `municipality_norm`, `district_norm`, `region_norm`


### 2.5 Granularity Adjustment

To prepare for schema integration, all datasets must share the same level of spatial granularity.

- **Parish-level datasets**: `short_term_rental_density`, `rental_prices`
- **Municipality-level datasets**: `municipalities`, `population_density`, `property_listings`

To unify them:
- Extracted `MunicipalityCode` from `geocod`:
  - `str[:4]` for parish-level datasets
  - `str[-4:]` for municipality-level datasets
- Aggregated parish-level data using `.groupby("MunicipalityCode").mean()`

This ensures alignment of all datasets for further merging and analysis.


---

## 3. 🧹 Data Cleaning

### 3.1 Overview

In this section, we apply all the cleaning and normalization operations identified during the profiling phase.

This includes:
- Type conversion of indicator columns (`valor`)
- Removal of invalid or null entries
- Standardization of text fields (municipality, region, district)
- Extraction of `MunicipalityCode` from `geocod`
- Aggregation of parish-level data to municipality level


### 3.2 Type Conversion

All datasets containing the `valor` column will be converted to numeric format to ensure consistency and allow aggregations.


In [ ]:
rental_prices["valor"] = pd.to_numeric(rental_prices["valor"], errors="coerce")
population_density["valor"] = pd.to_numeric(population_density["valor"], errors="coerce")
short_term_rental_density["valor"] = pd.to_numeric(short_term_rental_density["valor"], errors="coerce")

### 3.3 Remove Invalid Price Entries

Rows in `property_listings` with missing `Price` will be removed to ensure reliable analysis.


In [ ]:
property_listings = property_listings[property_listings["Price"].notnull()]

### 3.4 Normalize Text Columns

The following fields will be normalized:
- `Municipality`, `District`, `Region`

Normalization steps:
- Remove accents
- Convert to lowercase
- Trim whitespace
- Store results in new columns ending with `_norm`


In [ ]:
!pip install unidecode

In [ ]:
from unidecode import unidecode

def normalize_text(text):
    if pd.isnull(text):
        return ""
    return unidecode(str(text)).lower().strip()

# Normalize text in municipalities dataset
municipalities["Municipality"] = municipalities["CONCELHO_DSG"]
municipalities["Region"] = municipalities["NUTSIII_DSG"]
municipalities["municipality_norm"] = municipalities["Municipality"].apply(normalize_text)
municipalities["region_norm"] = municipalities["Region"].apply(normalize_text)

# Normalize property_listings
property_listings["city_norm"] = property_listings["City"].apply(normalize_text)
property_listings["district_norm"] = property_listings["District"].apply(normalize_text)

# Normalize geodsg in indicator datasets
rental_prices["municipality_norm"] = rental_prices["geodsg"].apply(normalize_text)
population_density["municipality_norm"] = population_density["geodsg"].apply(normalize_text)
short_term_rental_density["municipality_norm"] = short_term_rental_density["geodsg"].apply(normalize_text)

### 3.5 Extract and Unify MunicipalityCode

To merge datasets, we must ensure all contain the same municipality identifier.

- From `geocod` in parish-level datasets → extract **first 4 digits**
- From `geocod` in municipality-level datasets → extract **last 4 digits**


In [ ]:
rental_prices["MunicipalityCode"] = rental_prices["geocod"].astype(str).str[-4:]
population_density["MunicipalityCode"] = population_density["geocod"].astype(str).str[-4:]
short_term_rental_density["MunicipalityCode"] = short_term_rental_density["geocod"].astype(str).str[:4:]
municipalities["MunicipalityCode"] = municipalities["DICO"].astype(str)

### 3.6 Aggregate Parish-Level Data to Municipality

Since `rental_prices` and `short_term_rental_density` are at parish level, we aggregate them to municipality level using the average value per `MunicipalityCode`.

In [ ]:
rental_prices_avg = rental_prices.groupby("MunicipalityCode")["valor"].mean().reset_index(name="avg_rent_m2")
short_term_rental_density_avg = short_term_rental_density.groupby("MunicipalityCode")["valor"].mean().reset_index(name="avg_str_density")

Função para normalizar os nomes:

In [12]:
import unidecode

# Normalizar nomes
def normalizar(texto):
    return unidecode.unidecode(str(texto).strip().lower())

#### Areas_Freg_Conc_Dist_Pais_CAOP2019

Este ficheiro do CAOP fornece a divisão administrativa oficial de Portugal, com hierarquias de freguesia, concelho, distrito e NUTS. A folha Areas_Concelhos_CAOP2019 foi selecionada como base de referência para garantir consistência geográfica durante a integração dos dados.
- Inclui diversas colunas administrativas, nem todas relevantes para a análise;
- Os nomes dos concelhos e regiões precisam de ser normalizados;
- É necessário renomear colunas para maior clareza e coerência com os restantes datasets.

✅ Objetivos da Limpeza:
- Selecionar apenas colunas relevantes: `DICO`, `CONCELHO_DSG`, `NUTSIII_DSG`;
- Renomear colunas para `MunicipalityCode`, `Municipality`, `Region`;
- Normalizar nomes (`Municipality`, `Region`);
- Exportar dataset tratado como referência para junções (concelhosTratado.csv).

 1. Selecionar Colunas Relevantes

In [13]:
dfConcelhos = pd.DataFrame(concelhos)

colunasParaManter = ['DICO', 'CONCELHO_DSG', 'NUTSIII_DSG']
dfConcelhos = dfConcelhos[colunasParaManter]

2. Renomear Colunas com Nomes Descritivos

In [14]:
dfConcelhos = dfConcelhos.rename(columns={
    'DICO': 'MunicipalityCode',
    'CONCELHO_DSG': 'Municipality',
    'NUTSIII_DSG': 'Region'
})

 3. Normalizar Colunas de Texto

In [15]:
dfConcelhos['MunicipalityCode'] = dfConcelhos['MunicipalityCode'].astype(str).str.zfill(4)
dfConcelhos['Municipality_norm'] = dfConcelhos['Municipality'].apply(normalizar)
dfConcelhos['Region_norm'] = dfConcelhos['Region'].apply(normalizar)

> As colunas `Municipality_norm` e `Region_norm` serão usadas como auxilary keys nos joins entre datasets.

4. Exportar Tabela Tratada

In [16]:
dfConcelhosTratado = dfConcelhos[[
    'MunicipalityCode', 'Municipality', 'Municipality_norm',
    'Region', 'Region_norm'
]].drop_duplicates().copy()

dfConcelhosTratado.to_csv(r'dados\dados tradados/concelhosTratado.csv', index=False)

> A tabela `concelhosTratado.csv` será usada como base de referência para integrar os dados das rendas, densidade populacional, alojamentos e listings.

#### densidadealojamentosm2

Este dataset indica a densidade de alojamentos locais por km² ao nível da freguesia. Por estar numa granularidade inferior, é necessário ajustá-lo para concelho, a fim de permitir comparações consistentes com os restantes dados.
- Os dados estão organizados por geocod, que representa freguesias;
- Existem valores nulos em algumas entradas da densidade;
- Os nomes (`geodsg`) requerem normalização para integração.

✅ Objetivos da Limpeza:
- Extrair o código do concelho (`MunicipalityCode`) a partir do `geocod`;
- Normalizar os nomes das freguesias (`geodsg`);
- Substituir valores nulos pela média do concelho;
- Agregar a densidade média por concelho;
- Adicionar colunas normalizadas para integração.

1. Carregar Dados e Extrair Código do Concelho

In [17]:
dfAloj = pd.DataFrame(densidade_aloj)

# Extrair código do concelho (4 primeiros dígitos do geocod)
dfAloj['MunicipalityCode'] = dfAloj['geocod'].str[:4]

2. Juntar com Concelhos (para nomes normalizados)

In [18]:
# Juntar com concelhos para obter nomes e norm
dfAlojMerged = dfAloj.merge(
    dfConcelhos[['MunicipalityCode', 'Municipality', 'Municipality_norm']],
    on='MunicipalityCode', how='left'
)

3. Filtrar Matches Válidos

In [19]:
# Filtrar apenas matches válidos
alojamentosMatch = dfAlojMerged[dfAlojMerged['Municipality'].notna()].copy()

 4. Converter Densidade para float e Substituir Nulos

In [20]:
# Converter densidade para float
alojamentosMatch['Density'] = pd.to_numeric(alojamentosMatch['valor'], errors='coerce')

# Substituir valores nulos pela média do concelho
alojamentosMatch['Density'] = alojamentosMatch.groupby('MunicipalityCode')['Density'].transform(
    lambda x: x.fillna(x.mean())
)

5. Agregar Freguesias por Concelho (tratar da granularidade)

In [21]:
densidadeAlojMedia = (
    alojamentosMatch
    .groupby(['MunicipalityCode', 'Municipality', 'Municipality_norm'])
    .agg({'Density': 'mean'})
    .reset_index()
    .rename(columns={'Density': 'DensityAlojMean'})
)

6. Exportar Dataset Tradato

In [22]:
densidadeAlojMedia.to_csv(r'dados\dados tradados/densidadeAlojTratado.csv', index=False)

> A tabela final `densidadeAlojTratado.csv` contém a média da densidade de alojamentos por concelho, pronta para integração com os restantes datasets.

#### DensidadePopulacional

Este dataset apresenta a densidade populacional (número de habitantes por km²) por concelho, com dados já estruturados ao nível municipal, o que facilita a sua integração com os restantes datasets.
- Já inclui o código do concelho (`geocod`);

- Os nomes das regiões (`geodsg`) necessitam de normalização;

- A coluna valor deve ser convertida para numérica.

✅ Objetivos da Limpeza:

- Extrair `MunicipalityCode` a partir do `geocod`;

- Normalizar o nome da unidade geográfica (`geodsg`);

- Converter a densidade (`valor`) para float;

- Gerar colunas auxiliares (`Municipality`, `Municipality_norm`);

- Preparar o dataset final sem necessidade de agregação.

1. Extrair código do concelho (últimos 4 dígitos do geocod)

In [23]:
dfPop = pd.DataFrame(densidade_pop)

# 🧾 Extrair código do concelho (últimos 4 dígitos do geocod)
dfPop['MunicipalityCode'] = dfPop['geocod'].str[-4:]

2. Normalizar nome da unidade geográfica (`geodsg` → `Municipality_norm`)

In [24]:
dfPop['Municipality_norm'] = dfPop['geodsg'].apply(normalizar)
dfPop.rename(columns={'geodsg': 'Municipality'}, inplace=True)

3. Converter `valor` para float → coluna `Density`

In [25]:
dfPop['Density'] = pd.to_numeric(dfPop['valor'], errors='coerce')

4. Exportar CSV final tratado

In [26]:
dfPopTratado = dfPop[['geocod', 'Municipality', 'Municipality_norm', 'Density']]

# 💾 Exportar CSV final
dfPopTratado.to_csv(r'dados\dados tradados/densidadePopTratado.csv', index=False)


#### portugal_listings

Este dataset contém informações sobre os imóveis anunciados para aluguer em território nacional. Apesar de ser o dataset mais importante para o estudo do impacto do alojamento local, apresenta estrutura inconsistente com os restantes datasets, nomeadamente:
- Falta de código administrativo (geocod);
- Nomes de localidades em diferentes formatos (`District`, `City`, `Town`);
- Registos com preços inválidos (nulos ou 0).

✅ Objetivos da Limpeza:
- Remover colunas irrelevantes (`Unnamed`, `Town`);
- Converter Price para float, tratando entradas inválidas;
- Eliminar registos com preços ausentes ou inferiores a 1€;
- Normalizar os nomes das localidades (`District` e `City`);
- Substituir `City` por `Municipality`, para manter coerência com os restantes datasets;
- Gerar colunas auxiliares `*_norm` para usar como chaves durante o blocking e resolução de identidade.
> 🧠 Nesta fase, ainda não é possível adicionar `MunicipalityCode` ou `geocod`, visto que o dataset listings carece desta informação. A resolução de identidade com `dfConcelhosTratado` será feita na fase seguinte, através de blocking e semelhança textual.

1. Remover colunas irrelevantes

In [27]:
dfListings = pd.DataFrame(listings)

# 🧹 Remover colunas 'Unnamed' e 'Town' (não será usada)
dfListings = dfListings.drop(columns=[col for col in dfListings.columns if "Unnamed" in col])
dfListings = dfListings.drop(columns=['Town'], errors='ignore')

2. Converter `Price` para float e remover linhas com `Price` nulo ou menor que 0

In [28]:
# Converter 'Price' para float
dfListings['Price'] = pd.to_numeric(dfListings['Price'], errors='coerce')

# Filtrar linhas com preço válido (> 0)
dfListings = dfListings[dfListings['Price'].notna() & (dfListings['Price'] > 0)].copy()


3. Normalizar as colunas de texto e rename à coluna `City` para `Municipality`

In [29]:

dfListings['District_norm'] = dfListings['District'].apply(normalizar)
dfListings['Municipality'] = dfListings['City']  # Substitui City por Municipality
dfListings['Municipality_norm'] = dfListings['Municipality'].apply(normalizar)
dfListings = dfListings.drop(columns=['City'])

4. Exportar dados para tabela tradata

In [30]:
# 💾 Exportar CSV final tratado
dfListings.to_csv(r'dados\dados tradados/listingsTratado.csv', index=False)

#### rendasm2

Este dataset disponibiliza os valores médios das rendas por m<sup>2</sup> em Portugal, com granularidade ao nível da freguesia. No entanto, apresenta algumas limitações estruturais:
- Está ao nível da freguesia, sendo necessário agrupar por concelho;
- Algumas entradas têm valores nulos na coluna valor;
- Os nomes das regiões (coluna `geodsg`) podem apresentar variações e inconsistências linguísticas.

✅ Objetivos da Limpeza:
- Converter a coluna `valor` para o tipo float;
- Extrair o código do concelho (`MunicipalityCode`) a partir do `geocod`;
- Normalizar os nomes (`geodsg`);
- Substituir valores nulos pela média do concelho;
- Agregar os valores médios de renda ao nível municipal;
- Incluir colunas `Municipality` e `Municipality_norm` para integração.

1. Converter `valor` para float

In [31]:
dfRendas = pd.DataFrame(rendas)

# Converter 'valor' para float
dfRendas['RentValue'] = pd.to_numeric(dfRendas['valor'], errors='coerce')

2. Extrair código do concelho e normalizar nome

In [32]:

dfRendas['MunicipalityCode'] = dfRendas['geocod'].str[-4:]

# Normalizar nome do município
dfRendas['MunicipalityNorm'] = dfRendas['geodsg'].apply(normalizar)

3. Juntar com concelhos para validar correspondências

In [33]:
# Juntar com concelhos apenas para validar matches
rendasMerged = dfRendas.merge(
    dfConcelhos[['MunicipalityCode', 'Municipality', 'Municipality_norm']],
    on='MunicipalityCode', how='left'
)

4. Filtrar apenas correspondências válidas

In [34]:
# filtrar apenas matches válidos
rendasMatch = rendasMerged[rendasMerged['Municipality'].notna()].copy()

5. Substituir valores nulos pela média do concelho

In [35]:
# Substituir valores nulos pela média por concelho
rendasMatch['RentValue'] = rendasMatch.groupby('MunicipalityCode')['RentValue'].transform(
    lambda x: x.fillna(x.mean())
)

6. Agregar valor médio por concelho

In [36]:
# Agregar média por concelho
rendasPorConcelho = (
    rendasMatch
    .groupby('MunicipalityCode')
    .agg({'RentValue': 'mean'})
    .reset_index()
    .rename(columns={'RentValue': 'RentValueMean'})
)

7. Adicionar nomes normalizados ao resultado final

In [37]:
# Adicionar Municipality e MunicipalityNorm
rendasPorConcelho = rendasPorConcelho.merge(
    dfConcelhos[['MunicipalityCode', 'Municipality', 'Municipality_norm']],
    on='MunicipalityCode', how='left'
)

8. Exportar CSV final tratado

In [38]:
dfRendas = pd.DataFrame(rendasPorConcelho)

# Exportar CSV final
dfRendas.to_csv(r'dados\dados tradados/rendasTratado.csv', index=False)

---

## 🔗 Plano de Integração de Esquemas

Após o processo de limpeza e normalização dos dados, foi definido o seguinte plano para integrar os diversos datasets, tendo em conta as diferenças de granularidade e estrutura dos mesmos.

### 🧩 Objetivo da Integração

Integrar os datasets a um **nível municipal (MunicipalityCode)** para permitir a análise cruzada entre:
- Preços das rendas por m² (`rendasm2`)
- Densidade populacional (`densidadePopulacional`)
- Densidade de alojamentos locais (`densidadeAlojamentosm2`)
- Preços dos imóveis (`portugal_listings`)
- Localização administrativa (`Areas_Concelhos_CAOP2019`)


### 🗂️ Granularidade dos Dados

| Dataset                | Granularidade Original | Estratégia de Uniformização       |
|------------------------|------------------------|-----------------------------------|
| `densidadeAlojamentos` | Freguesia              | Agrupar por concelho com a média  |
| `rendasm2`             | Freguesia              | Agrupar por concelho com a média  |
| `densidadePopulacional`| Concelho               | Já está no nível correto          |
| `portugal_listings`    | Freguesia              | Usar a coluna concelho            |
| `Areas_Concelhos_CAOP2019` | Concelho           | Base de referência administrativa |


### 🧱 Chaves

- A **chave de integração principal** será `MunicipalityCode`, extraída de `geocod`, `DICO`, ou derivada de `City`.
- Para garantir consistência, foi criada a coluna `Municipality_norm` (nomes normalizados de concelhos).
- Os nomes de cidades (`City`) serão normalizados e mapeados para os concelhos correspondentes manualmente ou via regra de decisão (ex: correspondência direta com `Municipality_norm`).


### 📋 Estratégia de Integração

1. **Densidade de Alojamentos**:  
   - Agregar por `MunicipalityCode` (média dos valores da freguesia).
   - Gerar dataset final com: `MunicipalityCode`, `DensityAlojMean` e `Municipality_norm`.

2. **Densidade Populacional**:  
   - Já vem por `MunicipalityCode`, apenas normalizar e confirmar correspondência.

3. **Rendas por m²**:  
   - Agregar por `MunicipalityCode` (média dos valores da freguesia).
   - Gerar dataset com: `MunicipalityCode`, `RentValueMean` e `Municipality_norm`.

4. **Listings**:  
   - Associar `Municipality_norm` a `Municipality_norm` da tabela `concelhosTratado.csv` e obter `MunicipalityCode`.
   - Calcular estatísticas agregadas (média de preço, contagem) por concelho.


### 🧠 Considerações Finais

- O dataset `concelhosTratado.csv` será a **tabela base** para todas as junções.
- As colunas `MunicipalityCode` e `Municipality_norm` serão usadas como **blocking keys**.
- Pode ser necessário fazer **resolução de identidade** entre nomes de cidades (`City`) e concelhos (`Municipality`) para garantir boas ligações entre listings e os restantes datasets.

---

## 🧱 Blocking Strategy & Similarity Metrics

Durante a resolução de identidade e planeamento da integração, é essencial preparar mecanismos que reduzam o custo computacional das comparações entre registos (blocking) e definam critérios objetivos para medir semelhança (similarity metrics).

### 📦 Blocking Strategy

**Objetivo:** Reduzir o número de comparações entre entidades de diferentes datasets, criando "blocos" onde a comparação faz sentido (ex: concelhos com o mesmo prefixo).

#### Estratégia adotada:
- Criar blocos com base no prefixo do nome normalizado (`Municipality_norm`).
- Apenas comparamos nomes dentro do mesmo bloco.

### 📏 Similarity Metrics

Após o processo de blocking, é necessário aplicar métricas de semelhança para avaliar o quão próximos são dois nomes normalizados (`Municipality_norm`) de diferentes datasets. Para isso, foram utilizadas três métricas distintas da biblioteca `RapidFuzz`, que abordam a comparação textual sob diferentes perspetivas.

#### ✅ Métricas Utilizadas

| Métrica                   | Descrição                                                                 | Vantagens                                                                 |
|---------------------------|---------------------------------------------------------------------------|---------------------------------------------------------------------------|
| `token_sort_ratio`        | Ordena alfabeticamente as palavras antes da comparação                   | Lida bem com nomes invertidos, como `"porto vila"` vs `"vila porto"`      |
| `ratio` (Levenshtein)     | Mede a distância de edição entre duas strings                            | Capta pequenas variações e erros de digitação                             |
| `token_set_ratio`         | Compara os conjuntos únicos de palavras, ignorando repetições e ordem    | Útil quando um nome é subconjunto do outro: `"santa maria da feira"` vs `"feira"` |

Estas métricas são complementares e foram escolhidas por permitirem capturar:
- semelhança estrutural (`token_sort_ratio`);
- Variações simples de escrita (`ratio`);
- Inclusões e subconjuntos de palavras (`token_set_ratio`).

> 📌 A utilização conjunta destas métricas aumenta a robustez da correspondência entre nomes, facilitando a resolução de identidade.

In [39]:
!pip install rapidfuzz


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### 1. ✏️ Comparação 1: `listings` vs `rendas`

1.1 Carregar os datasets tratados

In [40]:
import pandas as pd
from rapidfuzz import fuzz

dfListings = pd.read_csv('dados/dados tradados/listingsTratado.csv')
dfRendas = pd.read_csv('dados/dados tradados/rendasTratado.csv')

> Objetivo: Importar os dois datasets que serão comparados — os listings e os valores de renda (rendas). Estes já foram limpos e normalizados anteriormente, incluindo a criação da coluna `Municipality_norm`.

1.2 Criar chave de blocking

In [41]:
dfListings['block_key'] = dfListings['Municipality_norm'].str[:4]
dfRendas['block_key'] = dfRendas['Municipality_norm'].str[:4]

> Objetivo: Reduzir o número de comparações utilizando uma chave de blocking.
>- A chave `block_key` é formada pelos primeiros 4 caracteres da string normalizada (`Municipality_norm`).
>- Assim, apenas localidades com prefixos semelhantes serão comparadas, o que melhora a performance e reduz falsos positivos.

1.3 Gerar pares candidatos com base no bloco

In [42]:
candidatos = pd.merge(
    dfListings[['Municipality', 'Municipality_norm', 'block_key']],
    dfRendas[['Municipality', 'Municipality_norm', 'block_key']],
    on='block_key',
    suffixes=('_listings', '_rendas')
)

> Objetivo: Criar todas as combinações possíveis dentro de cada bloco.
>- Une as duas tabelas pelos block_key.
>- O resultado é um conjunto de pares candidatos a representar a mesma entidade (concelho), mesmo que com variações no nome.

1.4 Calcular as 3 medidas de semelhança

In [43]:
candidatos['semelhança_token_sort'] = candidatos.apply(
    lambda row: fuzz.token_sort_ratio(row['Municipality_norm_listings'], row['Municipality_norm_rendas']) / 100,
    axis=1
)

candidatos['semelhança_levenshtein'] = candidatos.apply(
    lambda row: fuzz.ratio(row['Municipality_norm_listings'], row['Municipality_norm_rendas']) / 100,
    axis=1
)

candidatos['semelhança_token_set'] = candidatos.apply(
    lambda row: fuzz.token_set_ratio(row['Municipality_norm_listings'], row['Municipality_norm_rendas']) / 100,
    axis=1
)

> Objetivo: Avaliar o grau de semelhança entre os nomes dos concelhos nos dois datasets usando 3 métricas:
> -    `token_sort_ratio`: ordena as palavras antes de comparar, útil quando a ordem das palavras varia.
> -    `ratio (Levenshtein)`: mede a distância de edição entre as strings.
> -    `token_set_ratio`: ignora palavras repetidas e compara subconjuntos únicos, ideal quando há muitos termos comuns.
> Valores de `0` (menos similar) a `1` (mais similar).

1.5 Visualizar os pares com maior semelhança

In [44]:
top_pares = candidatos.sort_values(by='semelhança_token_sort', ascending=False)

display(top_pares[[
    'Municipality_listings', 'Municipality_rendas',
    'semelhança_token_sort',
    'semelhança_levenshtein',
    'semelhança_token_set'
]])

,Municipality_listings,Municipality_rendas,semelhança_token_sort,semelhança_levenshtein,semelhança_token_set
471810,Valpaços,VALPAÇOS,1.000000,1.000000,1.000000
0,Valpaços,VALPAÇOS,1.000000,1.000000,1.000000
471790,Moita,MOITA,1.000000,1.000000,1.000000
471789,Barreiro,BARREIRO,1.000000,1.000000,1.000000
471786,Montijo,MONTIJO,1.000000,1.000000,1.000000
...,...,...,...,...,...
82394,Santo Tirso,SANTA CRUZ DA GRACIOSA,0.242424,0.424242,0.242424
88687,Santo Tirso,SANTA CRUZ DA GRACIOSA,0.242424,0.424242,0.242424
95242,Santo Tirso,SANTA CRUZ DA GRACIOSA,0.242424,0.424242,0.242424
75912,Santo Tirso,SANTA CRUZ DA GRACIOSA,0.242424,0.424242,0.242424


> Objetivo: Ordenar os pares por semelhança mais alta (com base em `token_sort_ratio`) e apresentar os resultados mais promissores.
> - Permite analisar os pares com maior probabilidade de corresponderem à mesma entidade — essencial para fazer identity resolution entre `Municipality` do listings e `Municipality` dos valores de renda.

#### 2. ✏️ Comparação 2: `listings` vs `densidadePop`

2.1 Carregar ps datasets tratados

In [45]:
dfListings = pd.read_csv('dados/dados tradados/listingsTratado.csv')
dfPop = pd.read_csv('dados/dados tradados/densidadePopTratado.csv')

> **Objetivo:** Importar os datasets tratados para análise — neste caso, os listings e a densidade populacional (`densidadePop`).  
> Ambos foram previamente limpos e normalizados, incluindo a coluna `Municipality_norm`.

2.2 Criar chave de blocking

In [46]:
dfListings['block_key'] = dfListings['Municipality_norm'].str[:4]
dfPop['block_key'] = dfPop['Municipality_norm'].str[:4]

> **Objetivo:** Reduzir o número de comparações com base em blocos.  
> - A chave `block_key` é extraída dos primeiros 4 caracteres da `Municipality_norm`.  
> - Isso permite comparar apenas localidades com nomes semelhantes, reduzindo falsos positivos.

2.3 Gerar pares candidatos com base no bloco

In [47]:
candidatosPop = pd.merge(
    dfListings[['Municipality', 'Municipality_norm', 'block_key']],
    dfPop[['Municipality', 'Municipality_norm', 'block_key']],
    on='block_key',
    suffixes=('_listings', '_pop')
)

> **Objetivo:** Criar pares candidatos dentro de cada bloco.  
> - Junta ambos os datasets através do `block_key`.  
> - Resulta em pares com potencial para corresponder à mesma entidade.

2.4 Calcular as 3 medidas de semelhança

In [48]:
candidatosPop['semelhança_token_sort'] = candidatosPop.apply(
    lambda row: fuzz.token_sort_ratio(row['Municipality_norm_listings'], row['Municipality_norm_pop']) / 100,
    axis=1
)

candidatosPop['semelhança_levenshtein'] = candidatosPop.apply(
    lambda row: fuzz.ratio(row['Municipality_norm_listings'], row['Municipality_norm_pop']) / 100,
    axis=1
)

candidatosPop['semelhança_token_set'] = candidatosPop.apply(
    lambda row: fuzz.token_set_ratio(row['Municipality_norm_listings'], row['Municipality_norm_pop']) / 100,
    axis=1
)

> **Objetivo:** Avaliar semelhança textual entre os nomes das localidades.  
> - `token_sort_ratio`: útil se a ordem das palavras varia.  
> - `ratio`: mede a distância de edição.  
> - `token_set_ratio`: ideal quando há muitos termos comuns entre os nomes.

2.5 Visualizar os pares com maior semelhança

In [49]:
top_pares_pop = candidatosPop.sort_values(by='semelhança_token_sort', ascending=False)

display(top_pares_pop[[
    'Municipality_listings', 'Municipality_pop',
    'semelhança_token_sort',
    'semelhança_levenshtein',
    'semelhança_token_set'
]])

,Municipality_listings,Municipality_pop,semelhança_token_sort,semelhança_levenshtein,semelhança_token_set
494913,Valpaços,Valpaços,1.000000,1.000000,1.000000
0,Valpaços,Valpaços,1.000000,1.000000,1.000000
494894,Montalegre,Montalegre,1.000000,1.000000,1.000000
494893,Moita,Moita,1.000000,1.000000,1.000000
494892,Barreiro,Barreiro,1.000000,1.000000,1.000000
...,...,...,...,...,...
161261,Santo Tirso,Santa Cruz da Graciosa,0.242424,0.424242,0.242424
161077,Santo Tirso,Santa Cruz da Graciosa,0.242424,0.424242,0.242424
134729,Santo Tirso,Santa Cruz da Graciosa,0.242424,0.424242,0.242424
360334,Santo Tirso,Santa Cruz da Graciosa,0.242424,0.424242,0.242424


> **Objetivo:** Mostrar os pares com maior probabilidade de corresponderem à mesma entidade (`Municipality`).  
> Essencial para ligar listings à densidade populacional ao nível do concelho.

#### 3. ✏️ Comparação 2: `listings` vs `densidadeAloj`

3.1 Carregar os datasets tratados

In [50]:
dfListings = pd.read_csv('dados/dados tradados/listingsTratado.csv')
dfAloj = pd.read_csv('dados/dados tradados/densidadeAlojTratado.csv')

> 🧾 **Objetivo:** Importar os datasets normalizados — listings e densidade de alojamentos locais.

3.2 Criar chave de blocking

In [51]:
dfListings['block_key'] = dfListings['Municipality_norm'].str[:4]
dfAloj['block_key'] = dfAloj['Municipality_norm'].str[:4]

> **Objetivo:** Aplicar blocking para limitar as comparações a pares com prefixo semelhante.  
> Reduz o custo computacional e melhora a precisão da comparação.

3.3 Gerar pares candidatos com base no bloco

In [52]:
candidatosAloj = pd.merge(
    dfListings[['Municipality', 'Municipality_norm', 'block_key']],
    dfAloj[['Municipality', 'Municipality_norm', 'block_key']],
    on='block_key',
    suffixes=('_listings', '_aloj')
)

> **Objetivo:** Obter pares candidatos entre listings e densidade de alojamento.  
> Os pares são agrupados com base na chave de blocking.

3.4 Calcular as 3 medidas de semelhança

In [53]:
candidatosAloj['semelhança_token_sort'] = candidatosAloj.apply(
    lambda row: fuzz.token_sort_ratio(row['Municipality_norm_listings'], row['Municipality_norm_aloj']) / 100,
    axis=1
)

candidatosAloj['semelhança_levenshtein'] = candidatosAloj.apply(
    lambda row: fuzz.ratio(row['Municipality_norm_listings'], row['Municipality_norm_aloj']) / 100,
    axis=1
)

candidatosAloj['semelhança_token_set'] = candidatosAloj.apply(
    lambda row: fuzz.token_set_ratio(row['Municipality_norm_listings'], row['Municipality_norm_aloj']) / 100,
    axis=1
)

> **Objetivo:** Calcular a semelhança entre nomes para facilitar a correspondência automática.  
> As três métricas permitem complementar a análise em diferentes cenários de variação textual.

3.5 Visualizar os pares com maior semelhança

In [54]:
top_pares_aloj = candidatosAloj.sort_values(by='semelhança_token_sort', ascending=False)

display(top_pares_aloj[[
    'Municipality_listings', 'Municipality_aloj',
    'semelhança_token_sort',
    'semelhança_levenshtein',
    'semelhança_token_set'
]])

,Municipality_listings,Municipality_aloj,semelhança_token_sort,semelhança_levenshtein,semelhança_token_set
471810,Valpaços,VALPAÇOS,1.000000,1.000000,1.000000
0,Valpaços,VALPAÇOS,1.000000,1.000000,1.000000
471790,Moita,MOITA,1.000000,1.000000,1.000000
471789,Barreiro,BARREIRO,1.000000,1.000000,1.000000
471786,Montijo,MONTIJO,1.000000,1.000000,1.000000
...,...,...,...,...,...
82394,Santo Tirso,SANTA CRUZ DA GRACIOSA,0.242424,0.424242,0.242424
88687,Santo Tirso,SANTA CRUZ DA GRACIOSA,0.242424,0.424242,0.242424
95242,Santo Tirso,SANTA CRUZ DA GRACIOSA,0.242424,0.424242,0.242424
75912,Santo Tirso,SANTA CRUZ DA GRACIOSA,0.242424,0.424242,0.242424


> **Objetivo:** Analisar os melhores pares encontrados entre listings e densidade de alojamento local.  
> Essencial para realizar a correspondência automática entre os datasets.

### ✅ Conclusão da Resolução de Identidade

Com base na estratégia de **blocking por prefixo** e no uso de três métricas complementares de semelhança textual (`token_sort`, `token_set` e `Levenshtein`), foi possível gerar uma lista de pares candidatos com elevada probabilidade de corresponderem à mesma entidade (concelho), mesmo quando os nomes diferem ligeiramente entre datasets.

Este processo de **identity resolution** é fundamental para garantir a correta junção entre os dados dos `listings` e os restantes datasets (`rendasm2`, `densidadePopulacional`, etc.), assegurando que os valores agregados por concelho sejam coerentes e fiáveis.

As próximas etapas poderão incluir:
- Definir um **limiar de corte** para considerar pares como correspondentes (ex: ≥ 0.9 em `token_sort`);
- Realizar a **atribuição de `MunicipalityCode` ao dataset `listings`**, com base nos matches mais prováveis;
- Integrar os datasets agora harmonizados para análise estatística e visualização dos impactos territoriais.

Este processo completa a **Fase 1** do projeto, deixando os dados prontos para análise cruzada, correlações e visualizações interativas.


---

## 🧾 Conclusões e Trabalho Futuro

### ✅ Conclusões

- Os dados foram transformados e normalizados de forma a garantir **consistência semântica e estrutural** entre fontes com granularidade e formato distintos;
- Foi adotado um **modelo de integração baseado no concelho (`MunicipalityCode`)**, com o apoio de chaves auxiliares (`Municipality_norm`) para facilitar junções;
- Técnicas de **blocking e similarity metrics** permitiram gerar pares candidatos para a resolução de identidade, essencial para ligar dados do `portugal_listings` (sem `geocod`) aos restantes datasets georreferenciados;
- As três métricas escolhidas (Levenshtein, token_sort, token_set) mostraram-se eficazes na identificação de correspondências.

### 🔮 Trabalho Futuro

- **Definir limiares de confiança** para aceitar ou rejeitar pares com base nas medidas de semelhança (ex: ≥ 0.90);
- **Atribuir `MunicipalityCode` ao dataset `listings`** com base nos matches validados;
- **Integrar os datasets finais** para construir uma tabela única por concelho com: média de preços, densidade populacional, densidade de alojamentos e valor médio de renda;
- Aplicar **visualizações interativas** (ex: choropleths) para explorar o impacto do alojamento local;
- Avaliar possíveis **correlações** entre as variáveis — por exemplo, se maior densidade de listings está associada a aumento de preços ou densidade populacional.

---

> Esta preparação abre caminho para análises exploratórias, estatísticas e preditivas que respondam à questão central:  
**Qual o impacto da proliferação de Alojamentos Locais no custo e estrutura urbana do mercado de habitação em Portugal?**
